In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch4. OpenAI image API</font>

# OpenAI API를 활용한 이미지 생성 및 편집

- OpenAI의 gpt-image-1-mini 모델 API 활용 : 텍스트로부터 이미지 생성 및 기존 이미지 편집(Inpainting/Outpainting) 기능 구현

### 주요 학습 내용
1. gpt-image-1-mini 모델 개요 및 환경 설정
2. OpenAI API 클라이언트 생성
3. 기본 이미지 생성
4. 이미지 편집

# 1. gpt-image-1-mini 모델

- gpt-image-* : OpenAI가 개발한 최첨단 이미지 생성 AI 모델
    - 텍스트로 설명한 내용을 바탕으로 새로운 이미지를 생성

- 2023년에 공개된 DALL-E 2와 3 모델은 2026년 5월에 폐기됨

# 2. OpenAI API 클라이언트 생성

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

# 3. 기본 이미지 생성

## 3.1 이미지 생성 요청하기
- client.images.generate() 메서드 : 가장 간단한 형태로 이 메서드에 프롬프트 문자열만 전달하면 한 장의 이미지를 생성

In [ ]:
prompt_text = '숲이 보이는 밝은 자연 이미지를 그려줘'
model_img = 'gpt-image-1-mini'

In [ ]:
response = client.images.generate(
    model=model_img, # 모델 명칭
    prompt=prompt_text, # 생성할 이미지에 대한 설명
    n=1, # 생성할 이미지 수
    size='1024x1024' # 이미지 해상도 ; 기본값 : auto
)

In [ ]:
import base64
b64_data = response.data[0].b64_json

# base64 문자열을 실제 이미지 파일로 저장
image_bytes = base64.b64decode(b64_data)

with open('data/ch4_forest.png', 'wb') as f:
    f.write(image_bytes)
print('이미지 저장 완료')

In [ ]:
from IPython.display import display, Image
display(Image(image_bytes, width=200))

```
위 코드는 동일한 prompt에 quality="low" 옵션을 추가하여 이미지를 생성한 예시입니다. gpt-image-1-mini는 quality를 "low", "medium", "high", "auto"(기본값) 중에서 선택할 수 있으며, quality를 낮출수록 토큰 사용량과 비용이 줄어듭니다. 참고로 gpt-image-1-mini는 256x256, 512x512처럼 더 작은 정사각형 해상도는 지원하지 않으며, "1024x1024", "1536x1024", "1024x1536", "auto" 중에서만 크기를 선택할 수 있습니다(과거 DALL-E 2가 지원하던 256x256, 512x512 같은 작은 해상도는, DALL-E 2가 2026년 5월 API에서 제거되면서 함께 사라졌습니다).
```

> 참고: gpt-image-1-mini를 비롯한 GPT 이미지 계열 모델(gpt-image-1, gpt-image-1.5 등)은 URL이 아니라 항상 b64_json(base64로 인코딩된 이미지 데이터) 형태로만 결과를 반환합니다. 따라서 response.data[0].b64_json 값을 base64.b64decode()로 디코딩한 뒤 파일로 저장하거나 화면에 표시해야 합니다. (URL 방식은 과거 DALL-E 2, DALL-E 3에서 기본으로 제공되던 방식이었으나, 두 모델 모두 현재는 API에서 제거되어 더 이상 사용할 수 없습니다.)

In [ ]:
response = client.images.generate(
    model=model_img,
    prompt=prompt_text,
    n=1,
    size='1024x1024',
    quality='low' # 생성 퀄리티 low, medium, high, auto ; 기본값 : auto
)

In [ ]:
import base64
b64_data = response.data[0].b64_json

# base64 문자열을 실제 이미지 파일로 저장
image_bytes = base64.b64decode(b64_data)

with open('data/ch4_forest.png', 'wb') as f:
    f.write(image_bytes)
print('이미지 저장 완료')

## 3.2 quality 옵션으로 고품질 이미지 생성하기

```
앞서 언급했듯, OpenAI는 한때 DALL-E 2(기본값)와 DALL-E 3 두 가지 모델을 이미지 생성 API로 제공했지만, 두 모델 모두 2026년 5월 API에서 제거되었습니다. 현재는 gpt-image-1-mini, gpt-image-1, gpt-image-1.5 등 GPT 이미지 계열 모델만 사용할 수 있으며, 이 중 토큰당 비용이 가장 저렴한 모델이 gpt-image-1-mini입니다. DALL-E 3가 제공하던 "고품질" 옵션은 gpt-image-1-mini의 quality 파라미터로 대체할 수 있습니다.
```

In [ ]:
response = client.images.generate(
    model=model_img,
    prompt=prompt_text,
    n=1,
    size='1024x1024',
    quality='high'
)

In [ ]:
import base64
b64_data = response.data[0].b64_json

# base64 문자열을 실제 이미지 파일로 저장
image_bytes = base64.b64decode(b64_data)

with open('data/ch4_forest.png', 'wb') as f:
    f.write(image_bytes)
print('이미지 저장 완료')

In [ ]:
from PIL import Image
from io import BytesIO
image = Image.open(BytesIO(image_bytes))
image.resize((150,150))

> 주의: gpt-image-1-mini는 한 번의 API 호출로 n=1~10장까지 이미지를 생성할 수 있어, 여러 시안을 한 번에 비교할 때 유용합니다. 다만 지원하는 해상도는 "1024x1024"(정사각형), "1536x1024"(가로형), "1024x1536"(세로형), "auto"뿐이며, 과거 DALL-E 2가 지원하던 256x256, 512x512 같은 작은 해상도는 더 이상 생성할 수 없습니다. 작은 이미지가 필요하다면 1024x1024로 생성한 뒤 Pillow(PIL) 등으로 리사이즈해서 사용해야 합니다.

# 4. 이미지 편집

- 이미지 편집(edit) 기능
    - 인페인팅(Inpainting) : 이미지 내부 특정 영역의 내용 대체 및 수정 기술
    - 아웃페인팅(Outpainting) : 이미지 외부 빈 공간의 주변 맥락 기반 확장 생성 기술

## Inpainting - 이미지 부분 수정하기
- 필수 입력 데이터 : 편집 기준 원본 이미지, 수정 영역 지정 마스크 이미지
- 마스크(Mask) 이미지 규격 : 투명 영역(수정 대상)과 불투명 영역(유지 대상)으로 구성된 PNG 형식
- 처리 프로세스 : API(gpt-image-1-mini)에 원본·마스크 이미지 및 변경 프롬프트 동시 제공 후 자연스러운 영역 채움 수행
- 호출 메서드: client.images.edit()

In [ ]:
# 인페인팅
original_image = open('data/ch4_original.png', 'rb')
mask_image = open('data/ch4_mask.png', 'rb')

response = client.images.edit(
    model = model_img,
    image = original_image, # 편집할 이미지
    mask = mask_image,
    prompt = '하늘의 구름을 없애줘.',
    n = 1,
    size = '1024x1024'
)

In [ ]:
from IPython.display import display, Image
b24_data = response.data[0].b64_json

image_bytes = base64.b64decode(b24_data)
display(Image(image_bytes, width = 300))

```
위 코드에서 original.png는 편집 대상이 되는 원본 이미지이고, mask.png는 편집할 부분(하늘 영역)이 투명 처리된 마스크 이미지입니다. client.images.edit() 호출 시 prompt에 원하는 편집 내용을 서술하면 모델이 해당 부분을 반영한 이미지를 생성합니다. 결과는 이미지 생성 때와 마찬가지로 response.data[0].b64_json에 base64로 인코딩된 이미지 데이터가 반환됩니다(URL이 아님에 유의).
```

### 원본 이미지와 마스크 이미지에 대한 gpt-image-1-mini 요구사항
- 이미지: PNG, WEBP, JPG 중 하나의 형식이며 파일 크기 50MB 미만 (RGBA도 사용 가능하며, DALL-E 2처럼 굳이 RGB로 변환할 필요 없음)
- 마스크: 원본 이미지와 동일한 크기의 PNG 파일이며, 파일 크기 4MB 미만
- 마스크에서 완전히 투명한(alpha=0) 영역이 "편집할 부분"을 의미함 (DALL-E 2처럼 1비트 흑백(1 모드)으로 변환할 필요 없이, 알파 채널이 있는 RGBA 이미지를 그대로 사용하면 됨)
- 한 번의 요청에 원본 이미지를 최대 16장까지 함께 전달 가능 (DALL-E 2는 1장만 가능했음)